In [5]:
import yfinance as yf

# Get and save the historical data for the following tickers
tickers = [
    "ADBE",
    "AMD",
    "AXP",
    "XOM",
    "FDX",
    "HSY",
    "JPM",
    "HPE",
    "MRNA",
    "NVDA"
]

for each in tickers:
    
    # get ticker object
    tickr = yf.Ticker(each)
    
    # get historical data from 2015 - 2025
    data = tickr.history(start='2015-01-01', end='2025-01-01')

    data.to_csv(f"../data/raw/{each}.csv")

    print(f"Saved {each} data to ../data/raw/{each}.csv")


Saved ADBE data to ../data/raw/ADBE.csv
Saved AMD data to ../data/raw/AMD.csv
Saved AXP data to ../data/raw/AXP.csv
Saved XOM data to ../data/raw/XOM.csv
Saved FDX data to ../data/raw/FDX.csv
Saved HSY data to ../data/raw/HSY.csv
Saved JPM data to ../data/raw/JPM.csv
Saved HPE data to ../data/raw/HPE.csv
Saved MRNA data to ../data/raw/MRNA.csv
Saved NVDA data to ../data/raw/NVDA.csv


In [25]:
import pandas as pd

# Load each ticker data as its own dataframe

data = {}

for each in tickers:

    data[each] = pd.read_csv(f"../data/raw/{each}.csv")

data["NVDA"].head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2015-01-02 00:00:00-05:00,0.482423,0.486018,0.474754,0.482423,113680000,0.0,0.0
1,2015-01-05 00:00:00-05:00,0.482423,0.483861,0.472118,0.474275,197952000,0.0,0.0
2,2015-01-06 00:00:00-05:00,0.474994,0.475473,0.459416,0.459896,197764000,0.0,0.0
3,2015-01-07 00:00:00-05:00,0.463251,0.467325,0.457259,0.458697,321808000,0.0,0.0
4,2015-01-08 00:00:00-05:00,0.463970,0.478828,0.463730,0.475952,283780000,0.0,0.0


In [23]:
import matplotlib.pyplot as plt

# Plot ADBE, MRNA and NVDA close for full series
plot_tickers = ["ADBE","MRNA","NVDA"]

fig, axs = plt.subplots(figsize=(12,8), nrows=3, ncols=1, sharex=True)

for idx, ax in enumerate(axs):
    df = data[plot_tickers[idx]]
    ax.plot(df.Date, df.Close, label=plot_tickers[idx])
    ax.set_title(plot_tickers[idx])

    ticks = df.index[[0, len(df)//2, -1]]
    ax.set_xticks(ticks)

fig.supxlabel('Date')
fig.supylabel('Close')

fig.tight_layout()
#plt.show()
plt.close()

In [ ]:
# FEATURES
import numpy as np

# get 1 month 3 month 6 month and 12 month momentum ratio
for each in tickers:
    df = data[each]

    # momentum
    df["momentum_1m"] = df.Close / df.Close.shift(30) - 1
    df["momentum_3m"] = df.Close / df.Close.shift(90) - 1
    df["momentum_6m"] = df.Close / df.Close.shift(180) - 1
    df["momentum_12m"] = df.Close / df.Close.shift(360) - 1

    # compute the log returns and then volatility
    df["log_ret"] = np.log(df.Close / df.Close.shift(1))
    df["volatility"] = df["log_ret"].rolling(window=252).std() * np.sqrt(252)   # from formula sqrt(252) 

    # get rolling volume change
    df["volume_ratio"] = df.Volume / df.Volume.rolling(30).mean()

    # get fwd return 30 day
    df["target"] = df.Close.shift(-30) / df.Close - 1

data["HPE"].head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,momentum_1m,momentum_3m,momentum_6m,momentum_12m,log_ret,volatility,volume_ratio,target
0,2015-10-19 00:00:00-04:00,8.053967,8.053967,7.174560,7.205035,2721233,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.056798
1,2015-10-20 00:00:00-04:00,7.074429,7.370467,7.074429,7.344346,1220907,0.0,0.0,NaN,NaN,NaN,NaN,0.019151,NaN,NaN,-0.113811
2,2015-10-21 00:00:00-04:00,7.509777,7.553313,7.139731,7.174558,135236,0.0,0.0,NaN,NaN,NaN,NaN,-0.023390,NaN,NaN,-0.112864
3,2015-10-22 00:00:00-04:00,7.274691,7.422710,7.070077,7.379176,189261,0.0,0.0,NaN,NaN,NaN,NaN,0.028121,NaN,NaN,-0.102065
4,2015-10-23 00:00:00-04:00,7.509781,7.509781,7.335641,7.400944,103061,0.0,0.0,NaN,NaN,NaN,NaN,0.002946,NaN,NaN,-0.087881
